In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler

# Root folder holding the intermediate audio/video feature CSVs used below.
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

# videos
selected = pd.read_csv(os.path.join(DATA_ROOT, "ig_video_visualValid_N584_0901-1104_0325.csv"))

# audio_emo
audio_emo = pd.read_csv(os.path.join(DATA_ROOT, "audio_emotion_251128.csv"))
audio_emo = audio_emo.rename(columns={'filename': 'segment_name'})
audio_emo = audio_emo.rename(columns={'folder': 'Folder'})
audio_emo = audio_emo[["Folder", "segment_name", "angry", "disgust", "fearful", "happy", "neutral", "sad", "surprised"]]

# VUV
vuv = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "features_selected.csv"))
vuv = vuv[["Folder", "segment_name", "date", "name", "short_code", "VUV"]]

# pitch 
pitch = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "intonation_raw.csv"))
pitch['mean_f0'] = pitch[["Head", "Mid", "Tail"]].mean(axis=1)
pitch = pitch.rename(columns={'File': 'segment_name'})

# MFCC0
mfcc = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "ig_audios_2024_0901_1130_audios_features_mfcc0_new.csv"))
mfcc = mfcc.rename(columns={'file_name': 'Folder'})
mfcc = mfcc[['MFCC0', "Folder", "segment_name"]]

# Emotions
emotions = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "emotion_raw_0331.csv"))
emotions = emotions.rename(columns={'filename': 'segment_name'})
emotions = emotions[["arousal", "dominance", "valence", "Folder", "segment_name"]]
# Initialize MinMaxScaler with feature range (-1, 1)
scaler = MinMaxScaler(feature_range=(-1, 1))

# Fit and transform the data
emotions[['valence']] = scaler.fit_transform(emotions[['valence']])
emotions[['arousal']] = scaler.fit_transform(emotions[['arousal']])

# emotions['valence'] = emotions['valence'].apply(lambda x: x * 2.0 - 1.0)
# emotions['arousal'] = emotions['arousal'].apply(lambda x: x * 2.0 - 1.0)

df1 = vuv.merge(pitch, on = ["Folder", "segment_name"]).merge(mfcc, on = ["Folder", "segment_name"]).merge(emotions, on = ["Folder", "segment_name"]).merge(audio_emo, on = ["Folder", "segment_name"])
df1 = df1.rename(columns={'short_code': 'keycode'})

df1 = df1[df1['keycode'].isin(selected['keycode'])]

#formants
formants = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "formant_raw.csv"))
formants = formants.rename(columns={'folder': 'Folder', 'filename': 'segment_name'})

df2 = df1.merge(formants, on = ['Folder', 'segment_name', 'keycode', 'name'])

In [ ]:
# text = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "text_raw.csv"))

# def extract_name_keycode(folder_name):
#     parts = folder_name.split("_")
#     name = " ".join(parts[3:5])  # Extract full name
#     keycode = "_".join(parts[5:])
#     return name, keycode

# def get_segment_name(speaker, source_segment, sentence_id):
#     return f"{speaker}_{source_segment}_{sentence_id:02d}.wav"

# text[['name', 'keycode']] = text['Folder'].apply(lambda x: pd.Series(extract_name_keycode(x)))
# text['segment_name'] = text.apply(lambda row: get_segment_name(row['speaker'], row['source_segment'], row['sentence_id']), axis=1)

# for_text_emo = df2.merge(text, on = ['Folder', 'segment_name', 'keycode', 'name'])
# for_text_emo.to_csv(os.path.join(DATA_ROOT, "audio_features_raw_N9675_251113.csv"), index = False)

In [ ]:
# Select columns to compute weighted mean
columns_to_avg = ["VUV", "MFCC0", "mean_f0", "F1", "F2", "arousal", "dominance", "valence", 
                  "angry", "disgust", "fearful", "happy", "neutral", "sad", "surprised"]
weight_col = "Audio Length"

# Function to compute weighted mean
def weighted_mean(group):
    return (group[columns_to_avg].multiply(group[weight_col], axis=0).sum() / group[weight_col].sum())

# Group by Folder and compute weighted mean
df3 = df2.groupby(["Folder", "date", "name", "keycode"]).apply(weighted_mean).reset_index()

# Compute standard deviations (ignoring NaNs)
df3['std_f0'] = df1[df1["mean_f0"] != 0].groupby(["Folder", "date", "name", "keycode"])["mean_f0"].std().reset_index(drop=True)
df3['std_arousal'] = df1[df1["arousal"] != 0].groupby(["Folder", "date", "name", "keycode"])["arousal"].std().reset_index(drop=True)
df3['std_valence'] = df1[df1["valence"] != 0].groupby(["Folder", "date", "name", "keycode"])["valence"].std().reset_index(drop=True)
df3 = df3.fillna(0)


# text_emo -- produced by code/text emotion/model_load_emo.py
text_emo = pd.read_csv(os.environ.get("TEXT_EMO_CSV", "../../dataset/labeled_text_emo_N9675_sentenced_v5.csv"))
text_emo = text_emo[["Folder", "keycode", "anger_t", "disgust_t", "fear_t", "joy_t", "neutral_t", "sadness_t", "surprise_t"]]
text_emo1 = text_emo.groupby(["Folder", "keycode"]).mean().reset_index()

# Intonation
intonation = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "intonation_ratio.csv"))

# speaking rate
spr = pd.read_csv(os.path.join(DATA_ROOT, "audio_features", "speakingRate.csv"))
spr = spr[["Folder", "sentence_counts", "word_counts", "speaking_rate_w", "syll_counts", "speaking_rate_s", "text"]]

result = df3.merge(intonation, on = "Folder").merge(spr, on = "Folder").merge(text_emo1, on = ["Folder", "keycode"])

In [ ]:
win_dict = {
        "Bernie Moreno" : "W",
        "Bob Casey" : "L",
        "Colin Allred" : "L",
        "Dave McCormick": "W",
        "Kari Lake" : "L",
        "Ruben Gallego" : "W",
        "Sherrod Brown" : "L",
        "Ted Cruz" : "W",
        "Mike Rogers" : "L",
        "Elissa Slotkin" : "W",
        "Eric Hovde" : "L",
        "Tammy Baldwin" : "W",
    }
gender_dict =  {
        "Bernie Moreno" : "M",
        "Bob Casey" : "M",
        "Colin Allred" : "M",
        "Dave McCormick": "M",
        "Kari Lake" : "F",
        "Ruben Gallego" : "M",
        "Sherrod Brown" : "M",
        "Ted Cruz" : "M",
        "Mike Rogers" : "M",
        "Elissa Slotkin" : "F",
        "Eric Hovde" : "M",
        "Tammy Baldwin" : "F",
    }
party_dict = {
        "Bernie Moreno" : "R",
        "Bob Casey" : "D",
        "Colin Allred" : "D",
        "Dave McCormick": "R",
        "Kari Lake" : "R",
        "Ruben Gallego" : "D",
        "Sherrod Brown" : "D",
        "Ted Cruz" : "R",
        "Mike Rogers" : "R",
        "Elissa Slotkin" : "D",
        "Eric Hovde" : "R",
        "Tammy Baldwin" : "D",
    }

win_df = pd.DataFrame(list(win_dict.items()), columns=["name", "win"])
gender_df = pd.DataFrame(list(gender_dict.items()), columns=["name", "gender"])
party_df = pd.DataFrame(list(party_dict.items()), columns=["name", "party"])

results = result.merge(win_df, on="name", how="left")
results = results.merge(gender_df, on="name", how="left")
#results = results.merge(party_df, on="name", how="left")
results.to_csv(os.path.join(DATA_ROOT, "features_raw_251128.csv"), index=False)

In [ ]:
# Normalize likes and comments within each 'name'
# selected = pd.read_csv(os.path.join(DATA_ROOT, "ig_video_visualValid_N584_0901-1104_0325.csv"))

# # Replace -1 with NaN in a temporary column (to avoid modifying original data)
# selected['post_likes_clean'] = selected['post_likes'].replace(-1, np.nan)

# # Compute Z-score normalization, excluding NaN values
# selected['likes_z'] = selected.groupby('name')['post_likes_clean'].transform(lambda x: (x - x.mean()) / x.std())

# # Drop the temporary column if not needed
# selected.drop(columns=['post_likes_clean'], inplace=True)

# selected['comments_z'] = selected.groupby('name')['post_comments'].transform(lambda x: (x - x.mean()) / x.std())

# # Merge back with the original dataframe
# engagement = selected[['keycode', 'post_likes', 'post_comments', 'likes_z', 'comments_z', 'party', 'state']]

engagement = pd.read_csv(os.path.join(DATA_ROOT, "engagement", "ig_video_visualValid_N584_0901-1104_zscoreEngagement_0328.csv"))
#engagement = pd.read_csv(os.path.join(DATA_ROOT, "engagement", "ig_video_visualValid_N583_0901-1104_zscoreEngagement_250909.csv"))
merged = results.merge(engagement, on = ["keycode", "name"])

merged.to_csv(os.path.join(DATA_ROOT, "features_raw.csv"), index=False)

In [ ]:
columns_to_zscore = ["VUV", "MFCC0", "speaking_rate_s", "speaking_rate_w", "mean_f0", "std_f0", "F1", "F2", 
                     "angry", "disgusted", "fearful", "happy", "neutral", "sad", "surprised",
                     "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio", 
                     "arousal", "valence", "std_arousal", "std_valence"]

#va = merged.rename(columns={'arousal': 'arousal_raw', 'valence': 'valence_raw'})
va = results.rename(columns={'arousal': 'arousal_raw', 'valence': 'valence_raw'})
va = va[['arousal_raw', 'valence_raw']]

# by_video
results_zscore_v = results[columns_to_zscore].apply(zscore)
results_v = pd.concat([results.drop(columns=columns_to_zscore), results_zscore_v], axis=1)
results_v1 = pd.concat([results_v, va], axis=1)
results_v1.to_csv(os.path.join(DATA_ROOT, "features_video_251001.csv"))

# by_speaker
results_zscore_s = results.groupby(["name"])[columns_to_zscore].transform(zscore)
results_s = pd.concat([results.drop(columns=columns_to_zscore), results_zscore_s], axis=1)
results_s1 = pd.concat([results_s, va], axis=1)
results_s1.to_csv(os.path.join(DATA_ROOT, "features_speaker_251001.csv"))

In [ ]:
#df4.dropna(subset=["mean_f0"])
# df5[df5["std_f0"].isna()]

# # Check the size of each group
# group_sizes = df4.dropna(subset=["mean_f0"]).groupby(["Folder", "date", "name", "keycode"]).size()
# print(group_sizes)

# # Check the groups where the size is 1 (likely the issue)
# small_groups = group_sizes[group_sizes == 1]
# print(small_groups)

# # Optionally, you can also check the actual groups with only one value
# small_group_data = df4.dropna(subset=["mean_f0"]).groupby(["Folder", "date", "name", "keycode"]).filter(lambda x: len(x) == 1)
# print(small_group_data)


In [ ]:
features_raw = pd.read_csv(os.path.join(DATA_ROOT, "features_video_251001.csv"))
audio_emo = features_raw[["keycode", "name", "angry", "disgusted", "fearful", "happy", "neutral", "sad", "surprised"]]

file = pd.read_csv(os.path.join(DATA_ROOT, "experiment", "combined_240901-241104_N583_20250514.csv"))
merged = file.merge(audio_emo, on = ["keycode", "name"])

merged.to_csv(os.path.join(DATA_ROOT, "experiment", "combined_240901-241104_N583_20251001.csv"), index=False)